# WORD EMMBEDINGS

In [11]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords
import nltk
from openai import OpenAI
client = OpenAI()


In [12]:
## Leer los datos
df_tortugas = (pd.read_csv('../demo_datasets/demo_tortugas.csv', sep=';')
                #.head()
                .dropna(subset=['observa']))
df_tortugas.head()

,id,ficha,especie,nombre.especie,fecha_orig,fecha,anio,mes,estacion,lugar_orig,...,fmt_lugar,muni,codmun,causa_orig,causa,muerte,observa,lesion,cuerpo,estado
1,2,1868 - 1977-1990,Caretta caretta,Tortuga Boba,14/11/1989,14/11/1989,1989,Noviembre,Otoño,NaN,...,NaN,Icod de los Vinos,38022,Cautividad,Otros,No,3 años en cautividad en agua dulce.,NaN,NaN,NaN
2,3,9537 - 1998-2010,Caretta caretta,Tortuga Boba,02/12/2010,02/12/2010,2010,Diciembre,Invierno,CANDELARIA - CANDELARIA,...,"Candelaria, Santa Cruz de Tenerife, Islas Cana...",Candelaria,38011,Enfermedad,Enfermedad,No,"Le falta la aleta delantera dcha, caparazón y ...",Herida,Varias partes,NaN
3,4,9521 - 1998-2010,Caretta caretta,Tortuga Boba,15/11/2010,15/11/2010,2010,Noviembre,Otoño,Puerto Colón,...,"Puerto, Tazacorte, Santa Cruz de Tenerife, Isl...",Adeje,38001,Artes de pesca,Artes de pesca,Si,"Corte en el cuello por enmallamiento, flaca, d...",Varias lesiones,Cuello,NaN
4,5,9436 - 1998-2010,Caretta caretta,Tortuga Boba,29/09/2010,29/09/2010,2010,Septiembre,Otoño,EL PORIS - ARICO,...,"Carretera al Porís, 38588, Arico, Santa Cruz d...",Arico,38005,Artes de pesca,Artes de pesca,No,"aleta delantera dcha necrosada, jaime 15/10",NaN,Aleta,Necrosada
5,6,9433 - 1998-2010,Caretta caretta,Tortuga Boba,28/09/2010,28/09/2010,2010,Septiembre,Otoño,las teresitas,...,Playa de las Teresitas,Santa Cruz de Tenerife,38038,Artes de pesca,Artes de pesca,No,aleta delantera y trasera dchas con cortes por...,Varias lesiones,Aleta,NaN


## Descargar stopwords y seleccionar las de español

In [13]:
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jcge9\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
def extract_words(text):
    words = re.findall(r'\b[a-záéíóúñ]+\b', text.lower())
    # ❗ quitamos stopwords
    return [w for w in words if w not in stop_words]

## Construcción del vocabulario


In [15]:
words = []

for t in df_tortugas['observa']:
    words.extend(extract_words(t))

word_freq = Counter(words)

vocab = [w for w, f in word_freq.items() if f >= 1]

## Función para usar el modelo y obtener las embedding words

In [16]:
def get_embedding(text):
    return client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    ).data[0].embedding

word_emb = {w: get_embedding(w) for w in vocab}

## Buscar sinónimos de una palabra 

In [17]:

from sklearn.metrics.pairwise import cosine_similarity

def find_synonyms(word, word_emb, top_k=10):
    if word not in word_emb:
        return f"'{word}' no está en el vocabulario"

    target = np.array(word_emb[word]).reshape(1, -1)
    
    sims = []
    for w, emb in word_emb.items():
        if w == word:
            continue
        sim = cosine_similarity(
            target,
            np.array(emb).reshape(1, -1)
        )[0][0]
        sims.append((w, sim))
    
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:top_k]

## Ejemplos para buscar palabras relacionadas usando las word embeddings:

In [18]:
## Ejemplo de uso con palabra de lesiones corporales
palabras = ["lesion", 
            "hemorragia", 
            "fractura", 
            "contusión", 
            "corte"]

for palabra in palabras:
  print(f"""
> La palabras que se buscar es: {palabra}""")
  print(find_synonyms(palabra, word_emb))


> La palabras que se buscar es: lesion
'lesion' no está en el vocabulario

> La palabras que se buscar es: hemorragia
[('hematoma', np.float64(0.5750905739803378)), ('heridas', np.float64(0.499012544495892)), ('herida', np.float64(0.49279859149413063)), ('inflamación', np.float64(0.4384195548046934)), ('rafia', np.float64(0.4150664471739255)), ('necrótricas', np.float64(0.4032242590439975)), ('necrosada', np.float64(0.3850433332176139)), ('enredad', np.float64(0.38038589032560466)), ('amputación', np.float64(0.37156758844718285)), ('hombro', np.float64(0.3691332411653233))]

> La palabras que se buscar es: fractura
'fractura' no está en el vocabulario

> La palabras que se buscar es: contusión
'contusión' no está en el vocabulario

> La palabras que se buscar es: corte
[('cortes', np.float64(0.7332534932712393)), ('cuello', np.float64(0.4776447647752608)), ('amputación', np.float64(0.47020857547236206)), ('trozo', np.float64(0.46131317943389977)), ('crudo', np.float64(0.46079702899733

In [19]:
## Ejemplo de uso para la palabras de causas
palabras = ["causa", 
            "pesca", 
            "rafia", 
            "plastico", 
            "enrredada",
            "golpe"]

for palabra in palabras:
  print(f"""
> La palabras que se buscar es: {palabra}""")
  print(find_synonyms(palabra, word_emb))


> La palabras que se buscar es: causa
'causa' no está en el vocabulario

> La palabras que se buscar es: pesca
[('pescador', np.float64(0.7284549268044492)), ('peso', np.float64(0.5004523894896165)), ('percebes', np.float64(0.430575392811174)), ('anzuelo', np.float64(0.4001492183275682)), ('piel', np.float64(0.3732321745083283)), ('flota', np.float64(0.3697558858791583)), ('pequeña', np.float64(0.36027500557588377)), ('bajo', np.float64(0.3574197641293444)), ('flotando', np.float64(0.3557361195253522)), ('atada', np.float64(0.344911933826026))]

> La palabras que se buscar es: rafia
[('hemorragia', np.float64(0.4150664471739255)), ('putrefácta', np.float64(0.4147329247740538)), ('arrastraba', np.float64(0.40392455678620676)), ('crfs', np.float64(0.39056511979568054)), ('analítica', np.float64(0.36042195887594325)), ('crudo', np.float64(0.35695605787684)), ('restos', np.float64(0.35407316546437717)), ('roto', np.float64(0.3535768532884881)), ('inflamación', np.float64(0.348338004724925

In [20]:
## Ejemplo de uso para la palabras partes del cuerpo
palabras = ["cuerpo", 
            "cuello", 
            "capeza", 
            "aleta", 
            "caparazon"]

for palabra in palabras:
  print(f"""
> La palabras que se buscar es: {palabra}""")
  print(find_synonyms(palabra, word_emb))


> La palabras que se buscar es: cuerpo
[('cabeza', np.float64(0.6031261285878198)), ('cuello', np.float64(0.5226178033626623)), ('contenedor', np.float64(0.496216335167288)), ('parte', np.float64(0.4546921058719013)), ('huesos', np.float64(0.45084156147761506)), ('hombro', np.float64(0.44761087462402416)), ('casa', np.float64(0.42854220315754876)), ('derecho', np.float64(0.42093912848986653)), ('corte', np.float64(0.41986727974813004)), ('bajo', np.float64(0.4162435936407657))]

> La palabras que se buscar es: cuello
[('cabeza', np.float64(0.5867770942613237)), ('cuerpo', np.float64(0.5226178033626623)), ('nariz', np.float64(0.5148567133069704)), ('corte', np.float64(0.4776447647752608)), ('cortes', np.float64(0.47075012133962696)), ('codos', np.float64(0.4540674400615562)), ('hombro', np.float64(0.44546967853748376)), ('bajo', np.float64(0.4177111937287828)), ('encima', np.float64(0.41427974617170615)), ('anzuelo', np.float64(0.4021611741580977))]

> La palabras que se buscar es: cap